# Model & DB loading

In [3]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor
from FlagEmbedding import BGEM3FlagModel

# load the model and processor
ckpt_siglip2 = "google/siglip2-giant-opt-patch16-384"
ckpt_bgem3 = "BAAI/bge-m3"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor_siglip2 = AutoProcessor.from_pretrained(ckpt_siglip2)
model_siglip2 = AutoModel.from_pretrained(
    ckpt_siglip2,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" # Install flash-attn before using this
).eval()
model_bgem3 = BGEM3FlagModel(
    ckpt_bgem3,
    normalize_embeddings=True,
    use_bf16=True,
    devices=device,
    return_dense=True,
    return_sparse=True,
    # return_colbert_vecs=True,
    model_kwargs={"attn_implementation": "flash_attention_2"}
)

C:\Users\Bui Thien Nghia\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [4]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url='http://localhost:6333')

# Benchmark utils definition

In [17]:
def is_correct(candidate: list, ground_truth: list, tolerance=1):
    # [vid_id, frame_id]
    try:
        if candidate[0] != ground_truth[0]: return False
        return abs(int(candidate[1]) - int(ground_truth[1])) <= tolerance
    except Exception: return False

def calculate_metrics(candidate_list, ground_truth, k_list=[1, 5, 10, 50], tolerance=1):
    metrics = {f"Result@{k}": 0 for k in k_list}
    metrics["MRR"] = 0

    ranking = 0
    for i, candidate in enumerate(candidate_list):
        verdict = is_correct(candidate, ground_truth, tolerance)
        if not verdict: continue
        ranking = i + 1
        break

    if not ranking: return metrics

    metrics["MRR"] = 1 / ranking
    for k in k_list: metrics[f"Result@{k}"] = ranking <= k
    return metrics

def encode_query(query: str):
    inputs = processor_siglip2(text=query, padding='max_length', truncation=True, return_tensors='pt')

    with torch.no_grad():
        query_dense_siglip2 = model_siglip2.get_text_features(**inputs.to(device)).pooler_output
        query_dense_siglip2 = F.normalize(query_dense_siglip2, dim=-1)
    query_dense_siglip2 = query_dense_siglip2.to('cpu').squeeze().tolist()

    query_all_bgem3 = model_bgem3.encode(query)
    query_dense_bgem3 = query_all_bgem3['dense_vecs'].squeeze().tolist()
    query_sparse_bgem3 = models.SparseVector(
        indices=np.array(list(query_all_bgem3['lexical_weights'].keys()), dtype=np.int32).tolist(),
        values=np.array(list(query_all_bgem3['lexical_weights'].values()), dtype=np.float32).tolist(),
    )

    return query_dense_siglip2, query_dense_bgem3, query_sparse_bgem3

def do_query(query: str, collection_name: str, limit: int, prefetch_selection_index: list[int]):
    '''
        0: dense_siglip2\n
        1: dense_bgem3\n
        2: sparse_bgem3\n
    '''
    query_dense_siglip2, query_dense_bgem3, query_sparse_bgem3 = encode_query(query)

    prefetchs = [
        models.Prefetch(
            query=query_dense_siglip2,
            using="dense_siglip2",
            limit=limit * 2
        ),
        models.Prefetch(
            query=query_dense_bgem3,
            using="dense_bgem3",
            limit=limit * 2
        ),
        models.Prefetch(
            query=query_sparse_bgem3,
            using='sparse_bgem3',
            limit=limit * 2
        )
    ]

    selected_prefetchs = [prefetchs[i] for i in prefetch_selection_index]

    hits = client.query_points(
        collection_name=collection_name,
        prefetch=selected_prefetchs,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit
    ).points
    
    return hits

# Benchmark

In [18]:
import pandas as pd
from tqdm import tqdm

COLLECTION_NAME = 'test'
LIMIT = 100

COMBINATIONS = [{"prefetch_selections": comb, "benchmarks": []} for comb in [
        [0],
        [0, 1],
        [0, 2],
        [0, 1, 2],
    ]
]

test_set_path = "../../question_bank/questions_with_answers.csv"
test_set = pd.read_csv(test_set_path)
for i, row in tqdm(list(enumerate(list(test_set.itertuples())))):
    task = row.task
    vid_id = row.vid_id
    if task not in ['kis', 'qa', 'trake']:
        print(f"Test number {i + 1} got task \"{task}\" which is invalid")
        continue

    for combination in COMBINATIONS:
        selections = combination["prefetch_selections"]
        queries = row.query.split('|')
        frame_ids = row.frame_ids.split('|')

        for query, frame_id in zip(queries, frame_ids):
            hits = do_query(query, COLLECTION_NAME, LIMIT, selections)
            candidate_list = [[hit.payload['vid_id'], hit.payload['frame_id']] for hit in hits]
            # print(candidate_list)
            evaluation_result = calculate_metrics(candidate_list, [vid_id, frame_id])
            combination["benchmarks"].append(list(evaluation_result.values()))

100%|██████████| 20/20 [00:06<00:00,  3.22it/s]


In [20]:
import numpy as np

for combination in COMBINATIONS:
    scores = np.asanyarray(combination["benchmarks"])
    print(np.mean(scores, axis=0))

[0.34615385 0.80769231 0.84615385 0.88461538 0.54203024]
[0.26923077 0.65384615 0.80769231 0.84615385 0.42455521]
[0.26923077 0.69230769 0.80769231 0.88461538 0.43686013]
[0.19230769 0.57692308 0.73076923 0.84615385 0.34896747]
